# Record identifiers and external cohort flow

Run after `data_preparation.ipynb` and `evaluation.ipynb` (which writes `results_evaluation/patient_split.npz`).

1. Prints the external-cohort screening counts for Figure 1B.
2. Writes `appendix3_record_ids_split.csv` (ZZU pECG record and patient identifiers with the split assignment) for Multimedia Appendix 3.

In [ ]:
import os

import numpy as np
import pandas as pd

DATA_DIR = "rebuild_v3"
SAVE_DIR = "results_evaluation"

for name in ("korea_pairing_flow.csv", "korea_flow.csv"):
    print(f"\n{name}")
    print(pd.read_csv(os.path.join(DATA_DIR, name)).to_string(index=False))

In [ ]:
dev = np.load(os.path.join(DATA_DIR, "china_v2.npz"), allow_pickle=True)
split = np.load(os.path.join(SAVE_DIR, "patient_split.npz"), allow_pickle=True)

ids = dev["ids"].astype(str)
assignment = np.full(len(ids), "", dtype=object)
cv_fold = np.zeros(len(ids), dtype=int)
assignment[np.isin(ids, split["test_pts"].astype(str))] = "held-out test"
for k in range(1, 6):
    in_fold = np.isin(ids, split[f"val_pts_fold{k}"].astype(str))
    assignment[in_fold] = "train/validation"
    cv_fold[in_fold] = k
assert (assignment != "").all()

records = pd.DataFrame({
    "record_id": dev["ecg_ids"].astype(str),
    "patient_id": ids,
    "label": pd.Series(dev["y"]).map({0: "normal", 1: "hyperkalemia", 2: "hypokalemia"}).to_numpy(),
    "split": assignment,
    "cv_fold": cv_fold,
})
records.to_csv("appendix3_record_ids_split.csv", index=False)
print(records.groupby(["split", "label"]).size().unstack(fill_value=0))
print(records.groupby("split")["patient_id"].nunique())